# 07. InfraScan Road Damage Detection Training (YOLO on RDD2022)

This notebook automates training of YOLO object detectors on the RDD2022 road damage dataset.

**Key Specifications:**
- **Classes:** `0: D00 longitudinal crack`, `1: D10 transverse crack`, `2: D20 alligator crack`, `3: D40 pothole`
- **Models:** `yolov8s.pt` (Baseline) and `yolo11s.pt` (Comparison Architecture)
- **Standard Settings:** `imgsz=640`, `epochs=40`, `batch=16`, `patience=10`, `seed=42`, `workers=2`, `amp=True`, `cos_lr=True`, `flipud=0.0`
- **Storage:** Training data extracted to `/content` SSD for maximum I/O; weights & run cards persist to Google Drive (`save_period=-1`: only `best.pt` and `last.pt`)
- **Resume:** Automatically resumes from `last.pt` if a previous run was interrupted.

### 1. Verify GPU Hardware and CUDA Acceleration

Ensure Google Colab has assigned a CUDA-enabled GPU (typically NVIDIA T4).

In [ ]:
import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[!] WARNING: GPU not detected. Navigate to Runtime > Change runtime type > T4 GPU.")

### 2. Mount Google Drive

Mount Google Drive to access training archives and persist model checkpoints.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
assert os.path.exists("/content/drive/MyDrive"), "Google Drive mount failed!"
print("[OK] Google Drive mounted successfully at /content/drive.")

### 3. Install Ultralytics and Detection Dependencies

Install the official Ultralytics YOLO library and verify version compatibility.

In [ ]:
!pip install -q ultralytics

import ultralytics
print(f"[OK] Ultralytics version: {ultralytics.__version__}")

### 4. Extract Dataset and Code to Local Colab SSD (/content)

Always extract training images to `/content` (local NVMe SSD) rather than reading from Google Drive directly to avoid severe Drive I/O bottlenecks.

In [ ]:
import zipfile
from pathlib import Path

# 1. Extract Detection Source Code
code_zip_candidates = [
    Path("/content/drive/MyDrive/infrascan/ml_code_detection.zip"),
    Path("/content/drive/MyDrive/infrascan/data/ml_code_detection.zip"),
    Path("/content/drive/MyDrive/ml_code_detection.zip"),
]
code_zip = next((p for p in code_zip_candidates if p.exists()), None)
if code_zip is None:
    raise FileNotFoundError("ml_code_detection.zip not found on Google Drive! Please upload it.")

print(f"Extracting code from: {code_zip}...")
with zipfile.ZipFile(code_zip, "r") as zf:
    zf.extractall("/content")

# 2. Extract Small RDD2022 Dataset
data_zip_candidates = [
    Path("/content/drive/MyDrive/infrascan/data/rdd2022_yolo_small.zip"),
    Path("/content/drive/MyDrive/infrascan/rdd2022_yolo_small.zip"),
    Path("/content/drive/MyDrive/rdd2022_yolo_small.zip"),
]
data_zip = next((p for p in data_zip_candidates if p.exists()), None)
if data_zip is None:
    raise FileNotFoundError("rdd2022_yolo_small.zip not found on Google Drive! Please upload it.")

data_extract_dir = Path("/content/rdd2022_yolo_small")
if not (data_extract_dir / "images").exists():
    print(f"Extracting dataset from: {data_zip}...")
    with zipfile.ZipFile(data_zip, "r") as zf:
        zf.extractall("/content")
else:
    print("[OK] Dataset already extracted at /content/rdd2022_yolo_small.")

print("[OK] Environment setup complete.")

### 5. Setup Python Path and Import Detection Modules

Verify that the InfraScan detection package imports cleanly from `/content`.

In [ ]:
import sys
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from ml.detection import config, train_yolo
print("[OK] Detection package loaded successfully.")
print(f"Classes: {config.CLASS_NAMES}")
print(f"Standard Image Size: {config.IMAGE_SIZE}")
print(f"Standard Epochs    : {config.EPOCHS}")

### 6. Generate Dataset Configuration (data.yaml)

Generate `data.yaml` with absolute Colab paths pointing to `/content/rdd2022_yolo_small`.

In [ ]:
data_yaml_path = train_yolo.make_data_yaml(
    data_root="/content/rdd2022_yolo_small",
    out_path="/content/rdd2022_yolo_small/data.yaml",
)
with open(data_yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

### 7. Sanity Debug Run (1 Epoch, 10% Data Fraction)

Run a fast debug check (1 epoch on 10% data) to verify the loss computation, GPU gradient flow, and logging without waiting.

In [ ]:
print("Starting Sanity Debug Run...")
debug_results = train_yolo.train(
    model_weights="yolov8s.pt",
    data_yaml=data_yaml_path,
    project_dir="/content/debug_detection",
    run_name="debug_run",
    debug=True,
)
print("[OK] Sanity debug run completed successfully!")

### 8. Full Training Run 1: YOLOv8s Baseline (40 Epochs, Seed 42)

Train the primary baseline model (`yolov8s.pt`) with full hyperparameters.
Checkpoints (`best.pt`, `last.pt`) and `run_card.json` are persisted to Google Drive.

In [ ]:
drive_project_dir = "/content/drive/MyDrive/infrascan/models/detection"

yolov8s_results = train_yolo.train(
    model_weights="yolov8s.pt",
    data_yaml=data_yaml_path,
    project_dir=drive_project_dir,
    run_name="yolov8s_rdd2022",
    debug=False,
)
print("[OK] YOLOv8s full training complete!")

### 9. Full Training Run 2: YOLO11s Comparison Model (40 Epochs, Seed 42)

Train the secondary comparison model (`yolo11s.pt`) under identical conditions, seed, and data split in a separate folder.

In [ ]:
yolo11s_results = train_yolo.train(
    model_weights="yolo11s.pt",
    data_yaml=data_yaml_path,
    project_dir=drive_project_dir,
    run_name="yolo11s_rdd2022",
    debug=False,
)
print("[OK] YOLO11s full training complete!")

### 10. Verify Saved Checkpoints and Run Cards on Google Drive

Ensure that both models saved `best.pt`, `last.pt`, and `run_card.json` on Google Drive.

In [ ]:
for run_name in ["yolov8s_rdd2022", "yolo11s_rdd2022"]:
    run_folder = Path(drive_project_dir) / run_name
    weights_dir = run_folder / "weights"
    run_card_file = run_folder / "run_card.json"
    print(f"\n=== Checking Run: {run_name} ===")
    print(f"  Weights directory exists: {weights_dir.exists()}")
    if weights_dir.exists():
        for w in sorted(weights_dir.iterdir()):
            print(f"    + {w.name:15}: {w.stat().st_size / (1024**2):.2f} MB")
    print(f"  Run card exists: {run_card_file.exists()}")
    if run_card_file.exists():
        import json
        with open(run_card_file, "r") as f:
            card = json.load(f)
        print(f"    Weights: {card.get('model_weights')}")
        print(f"    Epochs: {card.get('hyperparameters', {}).get('epochs')}")
        print(f"    Date UTC: {card.get('date_utc')}")